# Урок 05 - Агентен RAG


## Настройка

Този бележник демонстрира шаблона Agentic RAG (Retrieval-Augmented Generation) с помощта на Microsoft Agent Framework.

**Предварителни изисквания:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — вашият крайна точка за услугата Azure AI Search
- `AZURE_SEARCH_API_KEY` — вашият API ключ за Azure AI Search
- Конфигурирано внедряване на Azure OpenAI чрез променливи на средата
- Аутентикация в Azure CLI (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Какво е Agentic RAG?

Традиционният RAG следва фиксиран процес: извличане на документи, след което генериране на отговор. **Agentic RAG** отива по-далеч, като дава на агента автономия да решава **кога** и **как** да извлича информация.

С Agentic RAG агентът може:
- **Да реши** дали е необходимо извличане преди отговора на въпрос
- **Да избира** кой източник на данни или инструмент да използва
- **Да оценява** извлечените резултати и да извършва последващи извличания, ако първият опит е недостатъчен
- **Да комбинира** информация от няколко стъпки на извличане в последователен отговор

Това прави агента по-гъвкав и точен в сравнение със статичния процес за извличане и след това генериране.


## Създаване на инструмент за търсене

В Agentic RAG външните източници на данни са опаковани като **инструменти**, които агентът може да извиква при необходимост. Това позволява на агента да третира извличането като просто още едно действие, което може да предприеме, а не като задължителна стъпка.

По-долу дефинираме база знания за пътувания и я представяме като инструмент, който агентът може да използва за търсене на информация за дестинации.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## Създаване на RAG агента

Сега създаваме агент, който е инструктиран да **винаги извлича информация преди да отговори**. Агентът използва инструмента `search_travel_knowledge`, за да осигури своя отговор, базирайки се на базата знания, а не на собствените си обучителни данни.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Итеративно извличане — Моделът Maker-Checker

Ключово предимство на Agentic RAG е **итеративното извличане**. Агентът може да извърши няколко кръга търсене, за да провери, уточни или разшири първоначалните си находки — подобно на работен процес "maker-checker":

1. **Стъпка Maker**: Агентът извлича първоначална информация и съставя отговор.
2. **Стъпка Checker**: Агентът извършва допълнителни извличания, за да провери подробностите или да запълни пропуските.

По-долу агентът е попитан с въпрос, който изисква сравнение на няколко дестинации, което го подтиква да търси няколко пъти.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Резюме

В този урок научихте как да изградите система **Agentic RAG** с помощта на Microsoft Agent Framework:

- **Agentic RAG** позволява на агентите автономно да решават кога да извличат информация, правейки извличането динамично, а не фиксирано.
- **Инструменти като източници на данни**: Външни бази знания (като Azure AI Search) се опаковат като инструменти, които агентът може да извиква.
- **Итеративно извличане**: Моделът maker-checker позволява на агента да извършва няколко кръга на извличане — търсене, проверка и усъвършенстване — преди да предостави окончателен отговор.

В продукция бихте заместили паметната `TRAVEL_KNOWLEDGE_BASE` с реален индекс на Azure AI Search за обработка на извличане на пътни документи в голям мащаб.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:  
Този документ е преведен с помощта на AI преводаческа услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля, имайте предвид, че автоматичните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критично важна информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
